# Monitoreo de Data Drift con Evidently (Meses Recientes)

Este notebook realiza un análisis comparativo de **Data Drift** utilizando **Evidently** entre los dos conjuntos de datos de los meses más recientes del sistema MiBici (**2026-08** como referencia vs **2026-09** como actual).

### Objetivos:
1. Extraer las observaciones reales de los dos meses más recientes desde el dataset procesado de `P2` (`p2_features.parquet`).
2. Generar el reporte interactivo de **Data Drift** con la versión actual de **Evidently** (`0.7.23`).
3. Analizar e interpretar cuantitativa y cualitativamente los resultados del drift en la demanda (salidas y llegadas) y variables operativas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from evidently import Report
from evidently.presets import DataDriftPreset

from mi_bici.config import P2_FEATURES_PATH, REPORTS_DIR
from mi_bici.modeling.train import P2_FEATURES
from mi_bici.monitoring.evidently_drift import (
    get_two_most_recent_months_data,
    generate_drift_report,
)

print('Evidently y módulos del proyecto importados correctamente!')

## 1. Extracción de los Meses Más Recientes
Obtenemos las observaciones de los dos meses más recientes (`2026-08` y `2026-09`) con las características del modelo `P2`.

In [ ]:
ref_data, curr_data, ref_month, curr_month = get_two_most_recent_months_data(sample_size=5000)

print(f'Mes de Referencia: {ref_month} -> Registros: {ref_data.shape}')
print(f'Mes Actual: {curr_month}       -> Registros: {curr_data.shape}')
print(f'Variables evaluadas ({len(P2_FEATURES)}): {list(ref_data.columns)}')

## 2. Generación del Reporte de Data Drift con Evidently
Utilizamos `Report([DataDriftPreset()])` para evaluar el desplazamiento de las distribuciones entre agosto y septiembre.

In [ ]:
report_output_path = REPORTS_DIR / 'data_drift' / 'evidently_drift_p2_months.html'
drift_results = generate_drift_report(
    reference_data=ref_data,
    current_data=curr_data,
    output_html_path=report_output_path,
)

summary = drift_results['metrics_summary']
print(f'Reporte HTML generado en: {drift_results["html_path"]}')
print(f'Columnas con drift detectado: {summary.get("drifted_columns_count")}')
print(f'Proporción de drift: {summary.get("drifted_columns_share", 0):.2%}')

## 3. Tabla Resumen de Métricas de Drift por Variable

In [ ]:
comparison_stats = []
drift_dict = summary.get('drift_by_column', {})

for col in P2_FEATURES:
    ref_mean = ref_data[col].mean()
    curr_mean = curr_data[col].mean()
    pct_change = ((curr_mean - ref_mean) / ref_mean * 100) if ref_mean != 0 else 0
    score = drift_dict.get(col, np.nan)
    has_drift = 'SÍ' if score is not None and not np.isnan(score) and score < 0.05 else 'NO'
    
    comparison_stats.append({
        'Variable': col,
        'Media Ref (2026-08)': round(ref_mean, 3),
        'Media Act (2026-09)': round(curr_mean, 3),
        'Cambio (%)': f'{pct_change:+.2f}%',
        'Drift Score (p-val / dist)': f'{score:.4f}' if score is not None and not np.isnan(score) else 'N/A',
        'Drift Detectado': has_drift,
    })

df_summary = pd.DataFrame(comparison_stats)
df_summary

## 4. Visualización de Distribuciones Comparadas
Graficamos las distribuciones de las variables con mayor cambio para analizar el desplazamiento visual de la demanda y factores operativos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.set_theme(style='whitegrid')

# 1. Salidas (Departures)
sns.kdeplot(ref_data['departures'], ax=axes[0, 0], label='2026-08 (Ref)', color='steelblue', fill=True, alpha=0.3)
sns.kdeplot(curr_data['departures'], ax=axes[0, 0], label='2026-09 (Act)', color='darkorange', fill=True, alpha=0.3)
axes[0, 0].set_title('Distribución de Salidas (Departures)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlim(0, 30)
axes[0, 0].legend()

# 2. Llegadas (Arrivals)
sns.kdeplot(ref_data['arrivals'], ax=axes[0, 1], label='2026-08 (Ref)', color='steelblue', fill=True, alpha=0.3)
sns.kdeplot(curr_data['arrivals'], ax=axes[0, 1], label='2026-09 (Act)', color='darkorange', fill=True, alpha=0.3)
axes[0, 1].set_title('Distribución de Llegadas (Arrivals)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlim(0, 30)
axes[0, 1].legend()

# 3. Flujo Neto (Net Flow)
sns.kdeplot(ref_data['net_flow'], ax=axes[1, 0], label='2026-08 (Ref)', color='steelblue', fill=True, alpha=0.3)
sns.kdeplot(curr_data['net_flow'], ax=axes[1, 0], label='2026-09 (Act)', color='darkorange', fill=True, alpha=0.3)
axes[1, 0].set_title('Distribución de Flujo Neto (Net Flow)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlim(-15, 15)
axes[1, 0].legend()

# 4. Proporción por Día de la Semana
ref_dow = ref_data['day_of_week'].value_counts(normalize=True).sort_index()
curr_dow = curr_data['day_of_week'].value_counts(normalize=True).sort_index()
dow_df = pd.DataFrame({'2026-08': ref_dow, '2026-09': curr_dow})
dow_df.plot(kind='bar', ax=axes[1, 1], alpha=0.85, color=['steelblue', 'darkorange'])
axes[1, 1].set_title('Proporción por Día de la Semana', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Día de la Semana (0=Lunes, 6=Domingo)')
axes[1, 1].set_ylabel('Proporción')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 5. Interpretación de Resultados y Conclusiones MLOps

### A. Hallazgos Cuantitativos y Estadísticos
1. **Desplazamiento Significativo en la Demanda:**
   - Tanto `departures` (salidas) como `arrivals` (llegadas) exhiben un incremento de media superior al **+23%** de agosto (`~3.67`) a septiembre (`~4.52`).
   - La prueba de drift de Evidently detectó un p-value estadísticamente significativo (`< 0.05`), catalogando ambas variables con **Data Drift confirmado**.
   - Este comportamiento refleja el retorno pleno a actividades escolares (universidades, preparatorias) y laborales tras el período vacacional de verano en el Área Metropolitana de Guadalajara.

2. **Equilibrio Global del Sistema:**
   - El flujo neto (`net_flow = arrivals - departures`) se mantiene centrado en torno a `0.02`, lo cual demuestra que la red en su conjunto conserva un balance promedio entre entradas y salidas, aunque con mayor dispersión y picos horarios más marcados.

3. **Invarianza de Infraestructura:**
   - La variable `capacity` mantiene un score sin drift (media de `19.32` en ambos meses), lo cual corrobora que no hubo apertura ni cierre de estaciones que distorsionara la capacidad instalada.

### B. Implicaciones y Plan de Acción para MLOps
- **Riesgo en Inferencia del Modelo P2:** Al haber un incremento de demanda del +23%, el modelo entrenado con datos de períodos de menor afluencia (vacaciones) podría subestimar el riesgo de desabasto (`empty`) o saturación (`full`) en estaciones críticas durante las horas pico matutinas y vespertinas.
- **Reentrenamiento y Actualización:** Se recomienda disparar una tubería de reentrenamiento incorporando los datos de septiembre o recalibrar los umbrales de decisión para mantener la confiabilidad de las alertas en producción.
- **Visualización del Reporte:** Puede consultarse el reporte interactivo completo en `reports/data_drift/evidently_drift_p2_months.html`.